In [ ]:
pip install gradio

In [ ]:
%pip install grad-cam

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import gradio as gr

device = torch.device("cpu")

# 1. Define Model Architectures (Dynamically reading classes from the checkpoint)
def load_model_and_classes(weights_path):
    checkpoint = torch.load(weights_path, map_location=device)
    
    # Extract the classes the model was actually trained on
    classes = checkpoint['classes']
    num_classes = len(classes)
    
    # Build model to match the exact number of classes
    model = models.mobilenet_v2(weights=None)
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Automatically generate the idx_to_class dictionary
    idx_to_class = {i: cls_name for i, cls_name in enumerate(classes)}
    
    return model, idx_to_class

# Load both models
model_gatekeeper, idx_to_class_stage1 = load_model_and_classes("gatekeeper_model.pth")
model_severity, idx_to_class_stage2 = load_model_and_classes("severity_model.pth")

print(f"Gatekeeper detects: {list(idx_to_class_stage1.values())}")
print(f"Severity model detects {len(idx_to_class_stage2)} sub-types: {list(idx_to_class_stage2.values())}")

# 2. Image Preprocessing Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Prediction & XAI Pipeline for Gradio UI
def analyze_lesion_ui(input_image):
    # Gradio passes a PIL Image directly
    raw_img = input_image.convert('RGB')
    input_tensor = transform(raw_img).unsqueeze(0)
    
    # Stage 1: Diagnosis
    with torch.no_grad():
        out_stage1 = model_gatekeeper(input_tensor)
        pred_idx1 = torch.argmax(out_stage1, dim=1).item()
        disease_pred = idx_to_class_stage1[pred_idx1]
    
    severity_pred = None
    target_model_for_cam = model_gatekeeper
    
    # Stage 2: Trigger Severity if Eczema
    if disease_pred == 'Eczema':
        with torch.no_grad():
            out_stage2 = model_severity(input_tensor)
            pred_idx2 = torch.argmax(out_stage2, dim=1).item()
            severity_pred = idx_to_class_stage2[pred_idx2]
        target_model_for_cam = model_severity  
    
    # Grad-CAM on the final convolutional layer of MobileNetV2
    target_layers = [target_model_for_cam.features[-1]]
    cam = GradCAM(model=target_model_for_cam, target_layers=target_layers)
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
    
    # Prepare background RGB image normalized between 0 and 1
    rgb_img = np.float32(raw_img.resize((224, 224))) / 255
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    
    # Build the Matplotlib figure
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(raw_img)
    
    title = f"Diagnosis: {disease_pred}"
    if severity_pred:
        title += f"\nSub-type / Severity: {severity_pred}"
        
    axes[0].set_title(title, fontsize=14, pad=10)
    axes[0].axis('off')
    
    axes[1].imshow(visualization)
    axes[1].set_title("Grad-CAM Explainability Heatmap", fontsize=14, pad=10)
    axes[1].axis('off')
    plt.tight_layout()
    
    # Return the figure object to Gradio
    return fig

# 4. Launch the Interactive Dashboard
# 4. Launch the Interactive Dashboard
demo = gr.Interface(
    fn=analyze_lesion_ui,
    inputs=gr.Image(type="pil", label="Upload Skin Lesion Image"),
    outputs=gr.Plot(label="Results"),
    title="Cascaded Dermatological Classifier",
    description="Upload a photo to diagnose Eczema, Melanoma, or Psoriasis. The secondary model will automatically classify the specific sub-type if Eczema is detected.",
    flagging_mode="never"  # <--- Updated parameter for newer Gradio versions
)

demo.launch()

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Define the Evaluation Function
def evaluate_model(model, test_dir, batch_size=16):
    # Use validation transforms (no augmentation, just resize and normalize)
    eval_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Load dataset
    test_dataset = datasets.ImageFolder(root=test_dir, transform=eval_transforms)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    class_names = test_dataset.classes
    print(f"Evaluating on classes: {class_names} (Total images: {len(test_dataset)})")
    
    y_true = []
    y_pred = []
    
    # 2. Run Predictions
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            
    # 3. Generate Classification Report
    print("\n--- Classification Report ---")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    # 4. Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('Confusion Matrix', fontsize=14)
    plt.show()

# --- Execution Setup ---
# 1. Ensure your model is loaded (using the load_model_and_classes function from your UI script)
# model_gatekeeper, _ = load_model_and_classes("gatekeeper_model.pth")

# 2. Set the path to a local folder containing your test images
# It must be structured like: test_data/Eczema/, test_data/Melanoma/, test_data/Psoriasis/
LOCAL_TEST_FOLDER = "D:\College Notes\TY Sem 5\PE_I\AISA MINIPROJECT\Data\Classifier" 

# 3. Run the evaluation
evaluate_model(model_gatekeeper, LOCAL_TEST_FOLDER)